# Variable-Channel Spatial ASR — Track C (공간 통합 ASR)

Qwen3-ASR-1.7B에 마이크 쌍 GCC-PHAT 공간 어댑터와 LoRA, 0.25초 격자 판정 층을 붙여 **한 모델이 누가 언제 무엇을 말했는지 한 번에** 냅니다. 설계는 [docs/unified-asr-design.md](https://github.com/hd1534/variable-channel-spatial-asr/blob/main/docs/unified-asr-design.md)에 있습니다.

| `CONFIG` | 하는 일 |
|---|---|
| `all` (기본값) | `track_c` → `track_c_mono` 순서로 학습·검증·본 평가 |
| `track_c` | 공간 어댑터가 있는 통합 모델 |
| `track_c_mono` | 같은 미세조정에서 공간 어댑터만 뺀 대조군. 두 설정의 차이가 공간의 몫입니다 |
| `smoke_real` | 실제 모델·실제 음성으로 30걸음만 돌려 메모리·속도·저장 경로를 확인 (점수는 의미 없음). 단어 정렬도 400발화만 |

- **순서**: 단어 정렬(처음 한 번) → 학습 → 검증 → 본 평가셋(`eval-t2`, `eval-t2-static`) 채점. 끝난 단계는 다시 실행해도 건너뜁니다.
- **관측 범위**: 20초 전체를 보고 답하므로 `full_audio` 참고 행입니다. VibeVoice·MOSS·Nemotron과 같은 조건입니다.
- **GPU**: A100 권장. 학습 메모리·속도는 `smoke_real`로 먼저 확인하세요.
- **백업**: 학습 노트북과 같은 개인 Drive 백업(`DRIVE_AUTH`)을 씁니다. 결과는 `runs/<설정>/`, 단어 정렬은 `trackc/alignments/`에 남습니다.

## 0. 설정

In [ ]:
STORAGE_BACKEND: str = "drive_api"  # @param ["drive_api", "mounted_drive"]
DRIVE_SECRET: str = "DRIVE_AUTH"  # @param {type:"string"}
LOCAL_STORE_ROOT: str = "/content/vcsasr-store"
BACKUP_EVERY_SEC: int = 120  # @param {type:"integer"}
# 이 VM으로 내려받지 않을 백업 경로 접두어 목록. None이면 기본값을 쓰고 []이면 모두 받습니다.
RESTORE_SKIP: list[str] | None = None
IMPORT_FROM: str = ""

CONFIG = "all"  # @param ["all", "track_c", "track_c_mono", "smoke_real"]
# 설정 덮어쓰기 (공백으로 구분, 값은 JSON). 예: max_steps=2000 batch_size=2
OVERRIDES = ""  # @param {type:"string"}
RUN_EVAL = True  # @param {type:"boolean"}
EVAL_LIMIT = 0  # @param {type:"integer"}
BRANCH = "main"  # @param {type:"string"}
COMMIT = ""  # @param {type:"string"}
ALLOW_CODE_CHANGE = False  # @param {type:"boolean"}

REPO_URL = "https://github.com/hd1534/variable-channel-spatial-asr.git"
DRIVE_ROOT: str = LOCAL_STORE_ROOT if STORAGE_BACKEND == "drive_api" else "/content/drive/MyDrive/vcsasr"
DATA_ROOT = "/content/data"
REPO_DIR = "/content/variable-channel-spatial-asr"
ENV_ROOT = "/content/vcsasr-envs"
SETS = ["eval-t2", "eval-t2-static"]

PRESETS = {"all": ["track_c", "track_c_mono"]}
CONFIG_LIST = PRESETS.get(CONFIG, [CONFIG])
print("실행할 Track C 설정:", CONFIG_LIST)

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from typing import Any

if "storage" not in globals():
    storage: Any | None = None


def run_local(cmd: str, check: bool = True) -> int:
    """백업 세션 없이 셸 명령을 실행한다. 백업을 연결하기 전의 git·pip 명령과 mounted_drive 모드에만 쓴다."""
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0 and check:
        raise RuntimeError(f"실패 (종료 코드 {proc.returncode}): {cmd}")
    return proc.returncode


def require_backup() -> None:
    """drive_api 모드에서 백업 세션 없이(예: 재연결 실패 후) 작업을 이어 가지 않도록 멈춘다."""
    if STORAGE_BACKEND == "drive_api" and storage is None:
        raise RuntimeError("개인 Drive 백업 세션이 없습니다. 연결 셀의 오류를 해결한 뒤 그 셀부터 다시 실행하세요.")


def sh(cmd: str, check: bool = True) -> int:
    """셸 명령을 실행하고 출력을 셀에 바로 보여 준다. 실패하면 셀을 멈춘다."""
    require_backup()
    if storage is None:
        return run_local(cmd, check)
    print("$", cmd, flush=True)
    return storage.run(cmd, shell=True, check=check)


def drop_repo_modules(repo_dir: str) -> int:
    """저장소 파일에서 불러온 모듈을 sys.modules에서 지워, 이후 import가 새로 체크아웃한 코드를 쓰게 한다."""
    root = Path(repo_dir).resolve()
    stale: list[str] = []
    for name, module in list(sys.modules.items()):
        try:
            paths = [getattr(module, "__file__", None), *(getattr(module, "__path__", None) or [])]
            inside = any(isinstance(p, str) and os.path.isabs(p) and Path(p).resolve().is_relative_to(root)
                         for p in paths)
        except Exception:  # 속성을 읽을 수 없는 특수 모듈은 저장소 모듈이 아니다.
            continue
        if inside and name != "__main__":
            stale.append(name)
    for name in stale:
        del sys.modules[name]
    return len(stale)


IN_COLAB = "google.colab" in sys.modules
import torch

GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(f"Colab: {IN_COLAB} | torch {torch.__version__} | GPU: {GPU or '없음'}"
      + (f" | compute capability {torch.cuda.get_device_capability()}" if GPU else ""))
if IN_COLAB and GPU is None:
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고른 뒤 다시 실행하세요.")

## 개인 Drive 연결 (처음 한 번)

학교 계정의 Colab Pro+를 유지하고 개인 계정의 Drive를 **Google 공식 Python 라이브러리**로 사용합니다.
[개인 Drive 설정 가이드](https://github.com/hd1534/variable-channel-spatial-asr/blob/main/docs/personal-drive.md)에 따라
Google Cloud 프로젝트에서 Drive API와 Desktop OAuth 클라이언트를 만들고, Mac에서 `python3 setup_drive.py`를 실행하세요.
생성된 `runs/private/drive-auth.json` 내용을 **Colab 왼쪽 열쇠 아이콘 → 보안 비밀 → `DRIVE_AUTH`**에 저장하고 노트북 접근을 허용합니다.
인증 파일·토큰은 채팅, 코드 셀, 출력, Git에 붙여 넣지 않습니다. 기존 rclone 설정은 사용하지 않습니다.

`STORAGE_BACKEND="drive_api"`가 기본입니다. 학교 Drive를 마운트하지 않으며, 파일은 VM의 `LOCAL_STORE_ROOT`에서 읽고 씁니다.
재접속하면 마지막 완료 백업을 검증해 복원합니다. 같은 계정의 일반 Drive 마운트는 `mounted_drive`로 선택할 수 있습니다.
OAuth 앱이 **Testing** 상태이면 갱신 토큰이 7일 뒤 만료되므로, 장기 사용 전 가이드의 게시 상태 안내도 확인하세요.

여러 런타임이 같은 백업 폴더를 동시에 써도 됩니다. 런타임마다 자기 백업 이력을 이어 가고, 연결 셀과 명령이 시작할 때마다 다른 런타임의 이력과 합칩니다.
한 런타임만 바꾼 폴더는 그 런타임의 버전을 씁니다. 같은 `runs/<실행>` 폴더를 두 런타임이 모두 바꿨다면 파일을 더 나중에 쓴 버전을 쓰고, 다른 버전은 백업 이력에 남깁니다(가이드의 `export --commit`으로 꺼냅니다). 그래서 이 노트북과 `benchmark.ipynb`를 서로 다른 런타임에서 동시에 돌려도 됩니다.
다만 같은 설정을 두 런타임에서 동시에 학습하지는 마세요. GPU 시간을 두 번 쓰고, 한쪽 결과는 최신 버전에서 빠집니다.
**이전 코드에서 올릴 때**: 이전 커밋(`9123e53`)의 코드로 돌고 있는 런타임은 다른 런타임이 백업을 올리면 다음 주기 백업에서 `remote advanced in another runtime` 오류로 작업을 멈춥니다. 새 코드의 연결 셀은 연결하자마자 백업을 올리므로, 새 코드로 다른 런타임을 연결하기 전에 이전 코드로 돌고 있는 런타임을 모두 멈추거나 새 코드로 다시 실행하세요.

`RESTORE_SKIP`은 이 VM으로 내려받지 않을 경로 접두어 목록입니다. 기본값(`None`)은 학습에 쓰지 않는 본 평가 결과(`eval`, `mono_baselines`), 전사 캐시, 공개 모델 가중치(`models/asr`, `models/huggingface`)를 건너뜁니다. 학습이 쓰는 ECAPA(`models/ecapa`)는 받습니다. `[]`이면 모두 받습니다. 건너뛴 파일도 백업에는 그대로 남습니다.

학교 Drive에 있던 기존 결과는 `IMPORT_FROM`(예: `/content/drive/MyDrive/vcsasr`)으로 한 번 가져옵니다. 연결 셀이 학교 Drive를 마운트하고, 개인 백업에 아직 없는 폴더(`runs/<실행>`, `eval/<이름>`, `rirbanks/<뱅크>`, `cache/` 파일 등)만 복사합니다. 이미 있는 폴더는 덮어쓰지 않으며(ASR 전사 캐시는 실행 계약과 세션 오디오 해시가 같을 때만 합치고, 다르면 백업의 캐시를 그대로 둡니다), 가져온 뒤에는 비워 둡니다. 가져오지 않으면 빈 백업에서 시작해 모든 시스템을 처음부터 학습합니다.

120초마다 학습 결과를 백업하고, 명령 종료 시 대용량 모델·코퍼스 캐시도 백업합니다.
업로드한 크기·체크섬을 확인한 뒤에만 완료 기록을 저장합니다. 전송 실패는 셀 오류로 표시합니다. 연결 셀이 실패하면 이후 명령 셀도 백업 없이 실행하지 않고 오류로 멈춥니다.
갑작스러운 런타임 종료 시 **마지막 완료 백업 이후 작업은 유실될 수 있습니다**. 마지막 셀의 `개인 Drive 최종 백업 완료`까지 확인하세요.

In [ ]:
if storage is not None:
    try:
        storage.close()
    except RuntimeError as exc:
        print(f"이전 백업 세션 종료: {exc}", flush=True)
    storage = None

# 백업 세션은 아래에서 연결하므로 그 전의 git·pip 명령은 run_local로 실행한다.
if not Path(REPO_DIR, ".git").is_dir():
    run_local(f'git clone -q --depth 1 --filter=blob:none --no-checkout -b {BRANCH} {REPO_URL} "{REPO_DIR}"')
    # Track C는 본 평가셋 오디오(eval-t2, eval-t2-static)로 채점한다. eval-dev 오디오만 뺀다.
    run_local(f"git -C \"{REPO_DIR}\" sparse-checkout set --no-cone '/*' '!/data/eval-dev/audio/'")
run_local(f'git -C "{REPO_DIR}" fetch -q --depth 1 origin {COMMIT or BRANCH} && git -C "{REPO_DIR}" checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
run_local("git log -1 --format='%h %s'")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# 같은 커널에서 새 커밋을 받았다면 이전 코드로 불러온 프로젝트 모듈을 비워, 아래 import부터 새 코드를 쓴다.
print(f"이전 코드로 불러온 프로젝트 모듈 {drop_repo_modules(REPO_DIR)}개를 비웠습니다.", flush=True)

if STORAGE_BACKEND == "drive_api":
    from drive_storage import connect_personal_drive
    run_local(f'{sys.executable} -m pip -q install -r requirements-drive.txt')
    if RESTORE_SKIP is None:
        # 다른 시스템의 평가 결과·전사 캐시·체크포인트와 공개 모델 가중치(Hugging Face에서 받는다)는 쓰지 않는다.
        RESTORE_SKIP = ["eval", "mono_baselines", "asr_cache", "asr_comparison.json", "models"] + [
            f"runs/{n}" for n in ("voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset",
                                  "proposed_carry_nodrop", "track_a", "track_a_spatial")]
    print("복원 제외 (백업에는 남음):", RESTORE_SKIP, flush=True)
    import_from: Path | None = None
    if IMPORT_FROM:
        if "google.colab" in sys.modules:
            from google.colab import drive
            drive.mount("/content/drive")
        import_from = Path(IMPORT_FROM)
        if not import_from.is_dir():
            raise FileNotFoundError(f"IMPORT_FROM 폴더가 없습니다: {IMPORT_FROM}")
        print(f"백업에 없는 결과 폴더를 {import_from}에서 가져옵니다.", flush=True)
    storage = connect_personal_drive(Path(LOCAL_STORE_ROOT), secret_name=DRIVE_SECRET, interval_sec=BACKUP_EVERY_SEC,
                                     skip=RESTORE_SKIP, import_from=import_from)
elif STORAGE_BACKEND == "mounted_drive":
    if IMPORT_FROM:
        raise ValueError('IMPORT_FROM은 STORAGE_BACKEND="drive_api"에서만 씁니다. mounted_drive는 마운트한 Drive를 바로 씁니다.')
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    raise ValueError("STORAGE_BACKEND를 확인하세요.")
print(f"저장소: {STORAGE_BACKEND} | 작업 경로: {DRIVE_ROOT}", flush=True)

## 1. Track C 환경

In [ ]:
# Track C 전용 venv: Colab의 CUDA PyTorch를 그대로 쓰고 qwen-asr·peft만 따로 설치한다.
sh("pip freeze | grep -iE '^(torch|torchaudio|numpy)==' > /content/constraints.txt")
from setup_asr_env import setup

TRACKC_PYTHON = str(setup("trackc", Path(ENV_ROOT)))
os.environ["VCSASR_MODEL_CACHE"] = str(Path(DRIVE_ROOT, "models", "asr"))
os.environ["HF_HUB_CACHE"] = str(Path(DRIVE_ROOT, "models", "huggingface"))
print("Track C 인터프리터:", TRACKC_PYTHON)

## 2. 데이터 (LibriSpeech, 방 음향 뱅크)

In [ ]:
LIBRISPEECH_MD5 = {
    "dev-clean": "42e2234ba48799c1f50f24a7926300a1",
    "train-clean-100": "2a93770f6d5c6c964bc36631d331a522",
}
MIRRORS = ["https://www.openslr.org/resources/12", "https://openslr.elda.org/resources/12",
           "https://openslr.trmal.net/resources/12"]

need = ["dev-clean", "train-clean-100"]

cache = Path(DRIVE_ROOT, "cache")
cache.mkdir(parents=True, exist_ok=True)
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
for name in need:
    if Path(DATA_ROOT, "LibriSpeech", name).is_dir():
        print("이미 풀려 있음:", name)
        continue
    cached = cache / f"{name}.tar.gz"
    if cached.is_file():
        tar = cached
    else:
        tar = Path(DATA_ROOT, f"{name}.tar.gz")
        for mirror in MIRRORS:
            if sh(f'curl -L --fail --retry 3 -C - -o "{tar}" {mirror}/{name}.tar.gz', check=False) == 0:
                break
        else:
            raise RuntimeError(f"{name}: OpenSLR 미러 세 곳 모두에서 받지 못했습니다.")
        digest = hashlib.md5()
        with open(tar, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 22), b""):
                digest.update(block)
        if digest.hexdigest() != LIBRISPEECH_MD5[name]:
            tar.unlink()
            raise RuntimeError(f"{name}: md5가 OpenSLR 공개값과 다릅니다. 받은 파일을 지웠으니 이 셀을 다시 실행하세요.")
    staging = Path(DATA_ROOT, f".extract-{name}")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    sh(f'tar -xzf "{tar}" -C "{staging}"')
    Path(DATA_ROOT, "LibriSpeech").mkdir(exist_ok=True)
    for item in Path(staging, "LibriSpeech").iterdir():
        target = Path(DATA_ROOT, "LibriSpeech", item.name)
        if not target.exists():
            item.rename(target)
    shutil.rmtree(staging, ignore_errors=True)
    if tar != cached:
        if shutil.disk_usage(DRIVE_ROOT).free > tar.stat().st_size + 2e9:
            shutil.copy2(tar, cached.with_name(cached.name + ".part"))
            os.replace(cached.with_name(cached.name + ".part"), cached)
        else:
            print(f"Drive 여유 공간이 부족해 {name} 압축 파일을 캐시하지 않습니다 (다음 세션에 다시 받습니다).")
        tar.unlink()


# 방 음향 뱅크는 다른 시스템과 같은 것을 쓴다 (Drive의 rirbanks에 있으면 그대로 받는다).
banks_ok = sh(f'{sys.executable} train.py --config configs/proposed_carry.json --data-root "{DATA_ROOT}" '
              f'--run-dir /content/.bankcheck --bank-mirror "{DRIVE_ROOT}/rirbanks" --check-banks', check=False)
if banks_ok == 3:
    sh('pip -q install -c /content/constraints.txt '
       '"pyroomacoustics @ git+https://github.com/LCAV/pyroomacoustics@v0.8.5"')

## 3. 단어 정렬 (Qwen3-ForcedAligner-0.6B, 처음 한 번)

In [ ]:
# 단어 정렬: 처음 한 번만 오래 걸린다. 끊겨도 다시 실행하면 이어서 한다.
ALIGN_DIR = Path(DRIVE_ROOT, "trackc", "alignments")
ALIGNMENTS = [ALIGN_DIR / "train-clean-100.jsonl", ALIGN_DIR / "dev-clean.jsonl"]
limit = ["--limit", "400"] if CONFIG_LIST == ["smoke_real"] else []
sh(" ".join([TRACKC_PYTHON, "-m", "trackc.align", "--corpus", f'"{DATA_ROOT}/LibriSpeech/train-clean-100"',
             "--out", f'"{ALIGNMENTS[0]}"', "--device", "cuda", *limit]))
sh(" ".join([TRACKC_PYTHON, "-m", "trackc.align", "--corpus", f'"{DATA_ROOT}/LibriSpeech/dev-clean"',
             "--exclude-eval-split", "--out", f'"{ALIGNMENTS[1]}"', "--device", "cuda", *limit]))

## 4. 학습 · 검증 · 본 평가

In [ ]:
from trackc.model import TrackCConfig
from trackc.train import eval_is_complete, run_is_complete


def trackc_args(name: str) -> tuple[str, str, TrackCConfig]:
    raw = json.loads(Path(f"configs/trackc/{name}.json").read_text(encoding="utf-8"))
    tokens = OVERRIDES.split()
    for token in tokens:
        key, _, value = token.partition("=")
        raw[key] = json.loads(value)
    cfg = TrackCConfig.from_dict(raw)
    run_dir = f"{DRIVE_ROOT}/runs/{cfg.name}"
    args = (f'-m trackc.train --config configs/trackc/{name}.json --data-root "{DATA_ROOT}" --run-dir "{run_dir}" '
            f'--alignments {" ".join(chr(34) + str(a) + chr(34) for a in ALIGNMENTS)} --bank-mirror "{DRIVE_ROOT}/rirbanks" '
            + ("--allow-code-change " if ALLOW_CODE_CHANGE else "")
            + (("--override " + " ".join(tokens)) if tokens else ""))
    return run_dir, args, cfg


for name in CONFIG_LIST:
    run_dir, args, cfg = trackc_args(name)
    print("\n" + "=" * 78 + f"\n>>> [{cfg.name}] 학습·검증\n" + "=" * 78, flush=True)
    if run_is_complete(run_dir, cfg):
        print(f"[skip] {cfg.name}: 같은 설정으로 끝난 summary.json이 있습니다.")
    else:
        sh(f"{TRACKC_PYTHON} {args}")
    if RUN_EVAL:
        limit = EVAL_LIMIT or (1 if name == "smoke_real" else 0)
        if eval_is_complete(run_dir, cfg, SETS, limit):
            print(f"[skip] {cfg.name}: 현재 best.pt로 채점한 본 평가 결과가 있습니다.")
        else:
            sh(f"{TRACKC_PYTHON} {args} --eval --sets {' '.join(SETS)} --limit {limit}")

## 5. 본 평가 결과 (full_audio)

In [ ]:
for name in CONFIG_LIST:
    _, _, cfg = trackc_args(name)
    for set_name in SETS:
        for variant in ("full", "no_space"):
            path = Path(DRIVE_ROOT, "runs", cfg.name, "eval", set_name, variant, "score.json")
            if not path.is_file():
                continue
            rep = json.loads(path.read_text(encoding="utf-8"))
            cells = " | ".join(f"M={m}: Acc {v['speaker_attribution_accuracy']:.3f}, cpWER {v['cpwer']:.3f}"
                               for m, v in rep["by_M"].items())
            print(f"[{set_name}] {cfg.name}/{variant} (full_audio): {cells}")

## 6. 결과 요약 블록 — 이 출력을 그대로 복사해 붙여 넣어 주세요

In [ ]:
results = {}
for name in CONFIG_LIST:
    run_dir, _, cfg = trackc_args(name)
    entry = {}
    summary_path = Path(run_dir, "summary.json")
    if summary_path.is_file():
        s = json.loads(summary_path.read_text(encoding="utf-8"))
        entry["summary"] = {k: s.get(k) for k in ("trained_steps", "best_step", "use_spatial", "gpu", "gpu_max_mem_gb",
                                                  "wall_sec", "commit", "val")}
    for path in sorted(Path(run_dir, "eval").glob("*/*/score.json")):
        rep = json.loads(path.read_text(encoding="utf-8"))
        entry[f"{path.parents[1].name}/{path.parent.name}"] = {"all": rep["all"], "by_M": rep["by_M"],
                                                                "truncated_outputs": rep.get("truncated_outputs")}
    results[cfg.name] = entry
print("===== VC-SASR TRACK C RESULTS (paste this back) =====")
print(json.dumps(results, ensure_ascii=False))
print("===== END =====")
require_backup()
if storage is not None:
    storage.flush()
    print("개인 Drive 최종 백업 완료", flush=True)